# NewsCLIPpings Inspection



## Setup

Check file paths, readability, and size in MB.

In [22]:
import json
from pathlib import Path
from collections import Counter

import pandas as pd

working_directory = Path.cwd()
project_root = working_directory.parent if working_directory.name == "notebooks" else working_directory
raw_root = project_root / "data" / "raw" / "newsclippings"

train_path = raw_root / "train.json"
val_path = raw_root / "val.json"
test_path = raw_root / "test.json"
paths = {"train": train_path, "val": val_path, "test": test_path}

In [23]:
file_rows = []
for split, path in paths.items():
    exists = path.is_file()
    readable = False
    if exists:
        with path.open("r", encoding="utf-8") as handle:
            handle.read(1)
        readable = True
    file_rows.append({
        "split": split,
        "path": str(path.relative_to(project_root)),
        "exists": exists,
        "size_mb": round(path.stat().st_size / 1_000_000, 3) if exists else None,
        "readable": readable,
    })
file_table = pd.DataFrame(file_rows)
display(file_table)

,split,path,exists,size_mb,readable
0,train,data/raw/newsclippings/train.json,True,7.708,True
1,val,data/raw/newsclippings/val.json,True,0.762,True
2,test,data/raw/newsclippings/test.json,True,0.788,True


## Load Data

Load each split and check the top-level structure.

In [24]:
objects = {}
for split, path in paths.items():
    with path.open("r", encoding="utf-8") as handle:
        objects[split] = json.load(handle)

object_table = pd.DataFrame([
    {"split": split, "python_type": type(obj).__name__,
     "top_level_keys": list(obj) if isinstance(obj, dict) else "not a dictionary"}
    for split, obj in objects.items()
])
display(object_table)

,split,python_type,top_level_keys
0,train,dict,"[source_datasets, annotations]"
1,val,dict,"[source_datasets, annotations]"
2,test,dict,"[source_datasets, annotations]"


In [4]:
structure_rows = []
for split, obj in objects.items():
    if not isinstance(obj, dict):
        raise TypeError(f"{split}: expected a top-level dictionary; observed {type(obj).__name__}")
    for key, value in obj.items():
        structure_rows.append({
            "split": split, "top_level_key": key,
            "value_type": type(value).__name__,
            "item_count": len(value) if isinstance(value, (dict, list)) else None,
        })
structure_table = pd.DataFrame(structure_rows)
display(structure_table)

,split,top_level_key,value_type,item_count
0,train,source_datasets,dict,4
1,train,annotations,list,71072
2,val,source_datasets,dict,4
3,val,annotations,list,7024
4,test,source_datasets,dict,4
5,test,annotations,list,7264


In [25]:
records = {}
source_maps = {}
for split, obj in objects.items():
    if not isinstance(obj.get("annotations"), list):
        raise TypeError(f"{split}: annotations must be a list")
    if not isinstance(obj.get("source_datasets"), dict):
        raise TypeError(f"{split}: source_datasets must be a dictionary")
    invalid_rows = [index for index, row in enumerate(obj["annotations"]) if not isinstance(row, dict)]
    if invalid_rows:
        raise TypeError(f"{split}: non-dictionary annotations at indices {invalid_rows[:5]}")
    records[split] = obj["annotations"]
    source_maps[split] = obj["source_datasets"]

fields = list(dict.fromkeys(field for rows in records.values() for row in rows for field in row))
print("Observed annotation fields:", fields)

Observed annotation fields: ['id', 'image_id', 'similarity_score', 'source_dataset', 'falsified']


## Sample Records

Inspect the first train record and five sample rows.

In [26]:
if not records["train"]:
    raise ValueError("train: no annotations available for first-record inspection")
first_annotation = records["train"][0]
first_record_table = pd.DataFrame([
    {"field": field, "value": value, "python_type": type(value).__name__}
    for field, value in first_annotation.items()
])
display(first_record_table)
display(pd.DataFrame(records["train"][:5]))

,field,value,python_type
0,id,620701,int
1,image_id,620701,int
2,similarity_score,1,int
3,source_dataset,2,int
4,falsified,False,bool


,id,image_id,similarity_score,source_dataset,falsified
0,620701,620701,1.000000,2,False
1,620701,1179991,0.847127,2,True
2,1569820,1569820,1.000000,2,False
3,1569820,1583815,0.716449,2,True
4,1490080,1490080,1.000000,2,False


## Split Counts

Compare observed counts with the expected merged/balanced counts.

In [27]:
expected_counts = {"train": 71_072, "val": 7_024, "test": 7_264}
split_table = pd.DataFrame([
    {"split": split, "observed_rows": len(rows), "expected_rows": expected_counts[split],
     "matches_expected": len(rows) == expected_counts[split]}
    for split, rows in records.items()
])
display(split_table)

,split,observed_rows,expected_rows,matches_expected
0,train,71072,71072,True
1,val,7024,7024,True
2,test,7264,7264,True


## Class Balance

Count pristine, falsified, and invalid labels. Percentages use all rows.

In [28]:
def class_label(value):
    if type(value) is bool:
        return "falsified" if value else "pristine"
    return "INVALID_OR_MISSING"

class_rows = []
for split, rows in records.items():
    counts = Counter(class_label(row.get("falsified")) for row in rows)
    balanced = counts["pristine"] == counts["falsified"] and counts["pristine"] > 0 and counts["INVALID_OR_MISSING"] == 0
    for label in ("pristine", "falsified", "INVALID_OR_MISSING"):
        class_rows.append({
            "split": split, "class": label, "count": counts[label],
            "percentage": round(100 * counts[label] / len(rows), 2) if rows else None,
            "balanced": balanced,
        })
class_table = pd.DataFrame(class_rows)
display(class_table)

,split,class,count,percentage,balanced
0,train,pristine,35536,50.0,True
1,train,falsified,35536,50.0,True
2,train,INVALID_OR_MISSING,0,0.0,True
3,val,pristine,3512,50.0,True
4,val,falsified,3512,50.0,True
5,val,INVALID_OR_MISSING,0,0.0,True
6,test,pristine,3632,50.0,True
7,test,falsified,3632,50.0,True
8,test,INVALID_OR_MISSING,0,0.0,True


## Field Checks

Check field types, missing/null values, and non-null unique counts.

In [29]:
def value_key(value):
    return (type(value).__name__, json.dumps(value, sort_keys=True, ensure_ascii=False))

field_rows = []
for split, rows in records.items():
    for field in fields:
        present = [row[field] for row in rows if field in row]
        types = Counter(type(value).__name__ for value in present)
        null_count = sum(value is None for value in present)
        missing_count = len(rows) - len(present)
        field_rows.append({
            "split": split, "field": field, "types_seen": dict(types),
            "missing_count": missing_count, "null_count": null_count,
            "missing_or_null_count": missing_count + null_count,
            "unique_count": len({value_key(value) for value in present if value is not None}),
        })
field_table = pd.DataFrame(field_rows)
display(field_table)

,split,field,types_seen,missing_count,null_count,missing_or_null_count,unique_count
0,train,id,{'int': 71072},0,0,0,35536
1,train,image_id,{'int': 71072},0,0,0,59876
2,train,similarity_score,"{'int': 35536, 'float': 35536}",0,0,0,30816
3,train,source_dataset,{'int': 71072},0,0,0,4
4,train,falsified,{'bool': 71072},0,0,0,2
5,val,id,{'int': 7024},0,0,0,3512
6,val,image_id,{'int': 7024},0,0,0,5943
7,val,similarity_score,"{'int': 3512, 'float': 3512}",0,0,0,3055
8,val,source_dataset,{'int': 7024},0,0,0,4
9,val,falsified,{'bool': 7024},0,0,0,2


## ID Checks

Check ID reuse and equality by class. Repeated rows count occurrences beyond the first.

Do not use id, image_id, id equality, source_dataset, or similarity_score as predictive model features; they are dataset/construction metadata.

In [30]:
def valid_id(value):
    return type(value) is int and value >= 0

id_rows = []
for split, rows in records.items():
    id_counts = Counter(row["id"] for row in rows if valid_id(row.get("id")))
    image_counts = Counter(row["image_id"] for row in rows if valid_id(row.get("image_id")))
    valid_pairs = [row for row in rows if valid_id(row.get("id")) and valid_id(row.get("image_id"))]
    equal_count = sum(row["id"] == row["image_id"] for row in valid_pairs)
    id_rows.append({
        "split": split, "unique_id": len(id_counts), "unique_image_id": len(image_counts),
        "repeated_id_rows": sum(count - 1 for count in id_counts.values()),
        "repeated_image_id_rows": sum(count - 1 for count in image_counts.values()),
        "id_values_repeated": sum(count > 1 for count in id_counts.values()),
        "image_id_values_repeated": sum(count > 1 for count in image_counts.values()),
        "id_equals_image_id": equal_count,
        "id_differs_from_image_id": len(valid_pairs) - equal_count,
        "invalid_id_pairs": len(rows) - len(valid_pairs),
    })
id_table = pd.DataFrame(id_rows)
display(id_table)

,split,unique_id,unique_image_id,repeated_id_rows,repeated_image_id_rows,id_values_repeated,image_id_values_repeated,id_equals_image_id,id_differs_from_image_id,invalid_id_pairs
0,train,35536,59876,35536,11196,35536,11196,35536,35536,0
1,val,3512,5943,3512,1081,3512,1081,3512,3512,0
2,test,3632,6147,3632,1117,3632,1117,3632,3632,0


In [31]:
relationship_rows = []
for split, rows in records.items():
    counts = Counter()
    for row in rows:
        relation = "invalid_id_pair"
        if valid_id(row.get("id")) and valid_id(row.get("image_id")):
            relation = "equal" if row["id"] == row["image_id"] else "different"
        counts[(class_label(row.get("falsified")), relation)] += 1
    for (label, relation), count in sorted(counts.items()):
        relationship_rows.append({"split": split, "class": label, "id_relationship": relation, "count": count})
relationship_table = pd.DataFrame(relationship_rows)
display(relationship_table)

,split,class,id_relationship,count
0,train,falsified,different,35536
1,train,pristine,equal,35536
2,val,falsified,different,3512
3,val,pristine,equal,3512
4,test,falsified,different,3632
5,test,pristine,equal,3632


## Duplicate Checks

Check exact rows and `(id, image_id, falsified)` combinations. Duplicate counts exclude the first occurrence.

In [32]:
duplicate_rows = []
for split, rows in records.items():
    exact_counts = Counter(json.dumps(row, sort_keys=True, ensure_ascii=False) for row in rows)
    combination_counts = Counter(
        tuple((field in row, value_key(row.get(field))) for field in ("id", "image_id", "falsified"))
        for row in rows
    )
    duplicate_rows.append({
        "split": split,
        "exact_duplicate_rows": sum(count - 1 for count in exact_counts.values()),
        "exact_duplicate_groups": sum(count > 1 for count in exact_counts.values()),
        "duplicate_combination_rows": sum(count - 1 for count in combination_counts.values()),
        "duplicate_combination_groups": sum(count > 1 for count in combination_counts.values()),
    })
duplicate_table = pd.DataFrame(duplicate_rows)
display(duplicate_table)

,split,exact_duplicate_rows,exact_duplicate_groups,duplicate_combination_rows,duplicate_combination_groups
0,train,0,0,0,0
1,val,0,0,0,0
2,test,0,0,0,0


## Source Distribution

Resolve source names and count rows by source and class.

In [33]:
source_lookup_table = pd.DataFrame([
    {"split": split, "source_dataset_key": key, "key_type": type(key).__name__,
     "source_name": name, "name_type": type(name).__name__}
    for split, mapping in source_maps.items() for key, name in mapping.items()
])
display(source_lookup_table)
print("Source dictionaries identical across splits:", source_maps["train"] == source_maps["val"] == source_maps["test"])


def valid_source(value, mapping):
    return type(value) is int and str(value) in mapping and type(mapping[str(value)]) is str and bool(mapping[str(value)].strip())


def source_name(value, mapping):
    return mapping[str(value)] if valid_source(value, mapping) else "UNKNOWN_OR_INVALID"

,split,source_dataset_key,key_type,source_name,name_type
0,train,0,str,semantics_clip_text_image,str
1,train,1,str,semantics_clip_text_text,str
2,train,2,str,scene_resnet_place,str
3,train,3,str,person_sbert_text_text,str
4,val,0,str,semantics_clip_text_image,str
5,val,1,str,semantics_clip_text_text,str
6,val,2,str,scene_resnet_place,str
7,val,3,str,person_sbert_text_text,str
8,test,0,str,semantics_clip_text_image,str
9,test,1,str,semantics_clip_text_text,str


Source dictionaries identical across splits: True


In [34]:
source_rows = []
for split, rows in records.items():
    counts = Counter()
    original_values = {}
    for row in rows:
        value = row.get("source_dataset")
        key = value_key(value)
        original_values[key] = value
        counts[(key, class_label(row.get("falsified")))] += 1
    for (key, label), count in sorted(counts.items()):
        value = original_values[key]
        source_rows.append({
            "split": split, "source_dataset": value, "source_name": source_name(value, source_maps[split]),
            "falsified": {"pristine": False, "falsified": True}.get(label, "INVALID_OR_MISSING"),
            "count": count,
        })
source_table = pd.DataFrame(source_rows)
display(source_table)

,split,source_dataset,source_name,falsified,count
0,train,0,semantics_clip_text_image,True,8884
1,train,0,semantics_clip_text_image,False,8884
2,train,1,semantics_clip_text_text,True,8884
3,train,1,semantics_clip_text_text,False,8884
4,train,2,scene_resnet_place,True,8884
5,train,2,scene_resnet_place,False,8884
6,train,3,person_sbert_text_text,True,8884
7,train,3,person_sbert_text_text,False,8884
8,val,0,semantics_clip_text_image,True,878
9,val,0,semantics_clip_text_image,False,878


## Similarity Scores

Summarize finite scores by class and source. Construction scores are not truth probabilities or necessarily comparable across strategies.

In [35]:
def finite_score(value):
    return type(value) in (int, float) and value == value and value not in (float("inf"), float("-inf"))


def score_summary(values):
    valid_values = [value for value in values if finite_score(value)]
    scores = pd.Series(valid_values, dtype="float64")
    return {
        "rows": len(values), "count": len(valid_values), "excluded_scores": len(values) - len(valid_values),
        "min": scores.min(), "max": scores.max(), "mean": scores.mean(), "median": scores.median(),
    }

score_class_rows = []
for split, rows in records.items():
    labels = sorted({class_label(row.get("falsified")) for row in rows})
    for label in labels:
        values = [row.get("similarity_score") for row in rows if class_label(row.get("falsified")) == label]
        score_class_rows.append({"split": split, "class": label, **score_summary(values)})
score_class_table = pd.DataFrame(score_class_rows)
display(score_class_table)

,split,class,rows,count,excluded_scores,min,max,mean,median
0,train,falsified,35536,35536,0,-0.047929,0.899991,0.616256,0.667147
1,train,pristine,35536,35536,0,1.000000,1.000000,1.000000,1.000000
2,val,falsified,3512,3512,0,0.158414,0.899893,0.617053,0.670353
3,val,pristine,3512,3512,0,1.000000,1.000000,1.000000,1.000000
4,test,falsified,3632,3632,0,0.105735,0.899964,0.619109,0.671759
5,test,pristine,3632,3632,0,1.000000,1.000000,1.000000,1.000000


In [36]:
score_source_rows = []
for split, rows in records.items():
    groups = {}
    source_values = {}
    for row in rows:
        value = row.get("source_dataset")
        key = (value_key(value), class_label(row.get("falsified")))
        source_values[key] = value
        groups.setdefault(key, []).append(row.get("similarity_score"))
    for key, values in sorted(groups.items()):
        value = source_values[key]
        score_source_rows.append({
            "split": split, "source_dataset": value, "source_name": source_name(value, source_maps[split]),
            "class": key[1], **score_summary(values),
        })
score_source_table = pd.DataFrame(score_source_rows)
display(score_source_table)

,split,source_dataset,source_name,class,rows,count,excluded_scores,min,max,mean,median
0,train,0,semantics_clip_text_image,falsified,8884,8884,0,0.186100,0.402687,0.300384,0.300932
1,train,0,semantics_clip_text_image,pristine,8884,8884,0,1.000000,1.000000,1.000000,1.000000
2,train,1,semantics_clip_text_text,falsified,8884,8884,0,-0.047929,0.871679,0.587581,0.599281
3,train,1,semantics_clip_text_text,pristine,8884,8884,0,1.000000,1.000000,1.000000,1.000000
4,train,2,scene_resnet_place,falsified,8884,8884,0,0.332369,0.899967,0.744896,0.767317
5,train,2,scene_resnet_place,pristine,8884,8884,0,1.000000,1.000000,1.000000,1.000000
6,train,3,person_sbert_text_text,falsified,8884,8884,0,0.481916,0.899991,0.832162,0.850291
7,train,3,person_sbert_text_text,pristine,8884,8884,0,1.000000,1.000000,1.000000,1.000000
8,val,0,semantics_clip_text_image,falsified,878,878,0,0.176689,0.394289,0.299290,0.299720
9,val,0,semantics_clip_text_image,pristine,878,878,0,1.000000,1.000000,1.000000,1.000000


## Split Overlap

Compare IDs across splits, including reuse across ID roles. Image near-duplicates remain unchecked.

In [37]:
id_sets = {
    split: {field: {row[field] for row in rows if valid_id(row.get(field))} for field in ("id", "image_id")}
    for split, rows in records.items()
}
overlap_rows = []
for left, right in (("train", "val"), ("train", "test"), ("val", "test")):
    for field in ("id", "image_id"):
        overlap_rows.append({
            "split_pair": f"{left} ∩ {right}", "id_type": field,
            "overlap_count": len(id_sets[left][field] & id_sets[right][field]),
        })
    left_union = id_sets[left]["id"] | id_sets[left]["image_id"]
    right_union = id_sets[right]["id"] | id_sets[right]["image_id"]
    overlap_rows.append({
        "split_pair": f"{left} ∩ {right}", "id_type": "either role (id ∪ image_id)",
        "overlap_count": len(left_union & right_union),
    })
overlap_table = pd.DataFrame(overlap_rows)
display(overlap_table)

,split_pair,id_type,overlap_count
0,train ∩ val,id,0
1,train ∩ val,image_id,0
2,train ∩ val,either role (id ∪ image_id),0
3,train ∩ test,id,0
4,train ∩ test,image_id,0
5,train ∩ test,either role (id ∪ image_id),0
6,val ∩ test,id,0
7,val ∩ test,image_id,0
8,val ∩ test,either role (id ∪ image_id),0


## Data Quality

Check missing values, types, IDs, labels, sources, scores, and duplicates.

In [38]:
allowed_types = {
    "id": (int,), "image_id": (int,), "similarity_score": (int, float),
    "source_dataset": (int,), "falsified": (bool,),
}
quality_rows = []
for split, rows in records.items():
    schema_fields = set(allowed_types)
    split_fields = field_table[field_table["split"] == split]
    duplicates = duplicate_table[duplicate_table["split"] == split].iloc[0]
    checks = {
        "missing annotation fields": int(split_fields["missing_count"].sum()),
        "null annotation fields": int(split_fields["null_count"].sum()),
        "unexpected field types (present, non-null)": sum(
            type(row[field]) not in types for row in rows for field, types in allowed_types.items()
            if field in row and row[field] is not None
        ),
        "rows with unexpected or absent schema fields": sum(set(row) != schema_fields for row in rows),
        "empty top-level structures": sum(isinstance(value, (dict, list)) and not value for value in objects[split].values()),
        "invalid or missing booleans": sum(type(row.get("falsified")) is not bool for row in rows),
        "NaN similarity values": sum(type(row.get("similarity_score")) is float and row["similarity_score"] != row["similarity_score"] for row in rows),
        "infinite similarity values": sum(type(row.get("similarity_score")) is float and row["similarity_score"] in (float("inf"), float("-inf")) for row in rows),
        "invalid or missing similarity values": sum(not finite_score(row.get("similarity_score")) for row in rows),
        "exact duplicate rows beyond first": int(duplicates["exact_duplicate_rows"]),
        "duplicate combinations beyond first": int(duplicates["duplicate_combination_rows"]),
        "unexpected or missing source_dataset values": sum(not valid_source(row.get("source_dataset"), source_maps[split]) for row in rows),
        "malformed or missing id": sum(not valid_id(row.get("id")) for row in rows),
        "malformed or missing image_id": sum(not valid_id(row.get("image_id")) for row in rows),
        "invalid source lookup entries": sum(
            not (type(key) is str and key.isdecimal() and type(name) is str and bool(name.strip()))
            for key, name in source_maps[split].items()
        ),
        "split size differs from supplied expectation": int(len(rows) != expected_counts[split]),
    }
    quality_rows.extend({"split": split, "check": check, "issue_count": count} for check, count in checks.items())
quality_table = pd.DataFrame(quality_rows)
display(quality_table.pivot(index="check", columns="split", values="issue_count").reindex(columns=list(records)))

split,train,val,test
check,,,
NaN similarity values,0,0,0
duplicate combinations beyond first,0,0,0
empty top-level structures,0,0,0
exact duplicate rows beyond first,0,0,0
infinite similarity values,0,0,0
invalid or missing booleans,0,0,0
invalid or missing similarity values,0,0,0
invalid source lookup entries,0,0,0
malformed or missing id,0,0,0


In [39]:
anomalies = quality_table[quality_table["issue_count"] > 0].reset_index(drop=True)
if anomalies.empty:
    print("No anomalies detected by the listed metadata quality checks.")
else:
    print("Observed issues — retain raw records and review before deciding the next step:")
    display(anomalies)

print("Raw records remain in memory only; no dataset, mapping, report, or asset was written.")

No anomalies detected by the listed metadata quality checks.
Raw records remain in memory only; no dataset, mapping, report, or asset was written.


## Notes

- Dataset structure and counts checked.
- No major metadata issues found.
- Next step will be decided after reviewing the results.